# Use your own SAE with SAR

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSAE/SAE-Behavioral-Compositionality/blob/main/notebooks/Use_Your_Own_SAE.ipynb)

This notebook has **two working paths**:

1. Use any locally loaded **SAE Lens or custom PyTorch SAE** on a compatible TransformerLens model. The default configuration is a public SAE so **Run all** works without your own files.
2. Connect a completely custom model and encoder using small NumPy callables, without TransformerLens or SAE Lens internals.

Both paths use exactly the same `evaluate_sae()` and unchanged `sar.py` implementation. This is **not** a claim that any model/SAE pair automatically works: the model, site, preprocessing and dictionary must match.

## 1 · Set up

The project is loaded from the public repository; local checkouts work too.

In [ ]:
# Environment and project code. On Colab, install known compatible versions.
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "transformer-lens==3.5.0", "sae-lens==6.44.4"])

# A Colab notebook opened from GitHub does not automatically contain repo files.
REPO_URL = "https://github.com/OpenSAE/SAE-Behavioral-Compositionality.git"
project = next((p for p in [Path.cwd(), Path.cwd().parent]
                if (p / "sar_runner.py").exists() and (p / "sar.py").exists()), None)
if project is None:
    project = Path("/content/SAE-Behavioral-Compositionality" if IS_COLAB else "./SAE-Behavioral-Compositionality")
    if not (project / "sar_runner.py").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(project)])
    if not (project / "sar_runner.py").exists():
        raise FileNotFoundError("sar_runner.py missing. Push the new files to the public repository first.")
sys.path.insert(0, str(project.resolve()))

import numpy as np
from sar import coalitions, design, sar
from sar_runner import (ActivationSite, FunctionalModelAdapter, FunctionalSAE,
                        HookedTransformerAdapter, LogitDifference, TorchSAEFeatures,
                        evaluate_sae)
print("SAR source:", project)
print("NumPy:", np.__version__)


## 2 · Real-model path: select your model and SAE

**Change the five configuration values below** to evaluate another matching checkpoint. GPT-2 and a publicly released SAE are used by default so this notebook is fully runnable. If you have an SAE instance with `.encode(torch.Tensor)` and `.W_dec` (shape `[n_features, d_model]`), set `MY_SAE` to that instance after loading it.

In [ ]:
# ---------------- Editable settings ----------------
MODEL_NAME = "gpt2"
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.6.hook_resid_pre"
HOOK_NAME = "blocks.6.hook_resid_pre"      # must identify the SAME site used for SAE training
PROMPT = "Alice Emma Alice"
TARGET, FOIL = " Emma", " Rachel"        # each string must be exactly ONE model token
MY_SAE = None                            # or assign a loaded custom PyTorch SAE here
K, M, N_FIT, BATCH = 8, 144, 104, 16
# ---------------------------------------------------

import torch
from sae_lens import SAE
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
if MY_SAE is None:
    loaded = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=DEVICE)
    MY_SAE = loaded[0] if isinstance(loaded, tuple) else loaded
if hasattr(MY_SAE, "eval"):
    MY_SAE.eval()
features = TorchSAEFeatures(MY_SAE)
print("Model device:", DEVICE, "SAE decoder shape:", tuple(MY_SAE.W_dec.shape))
if features.hook_name is not None:
    assert features.hook_name == HOOK_NAME, "SAE and hook metadata mismatch"


### Run the same intervention and SAR estimator

This checks that the unmodified full-coalition score agrees with the clean forward pass. When it does not, SAR will **raise an error** instead of silently reporting a misleading number. For unusual models or other hook sites use `run_mode="full"` rather than `"resume"`.

In [ ]:
import re
mode = "resume" if re.fullmatch(r"blocks\.\d+\.hook_resid_pre", HOOK_NAME) else "full"
runner = HookedTransformerAdapter(model, HOOK_NAME,
                                   behavior=LogitDifference(TARGET, FOIL), run_mode=mode)
result = evaluate_sae(runner, features, PROMPT,
                      K=K, M=M, n_fit=N_FIT, batch_size=BATCH, seed=7)
for key in ("sar1", "sar2", "rho2", "rank1", "rank2", "sign1", "sign2"):
    print(f"{key:>7}: {result.metrics[key]:.5f}")
print("Selected SAE features:", result.feature_ids.tolist())
print("Clean/full-coalition score:", round(result.full_response, 5))


## 3 · Fully custom SAE and model (no TransformerLens hooks)

Below is a complete **independent adapter example**, with a tiny synthetic model where the hidden activation and continuation are explicit. Replace `capture_my_model()` and `score_my_model()` with the methods that read/patch your model. Replace `encode_my_sae()` and the decoder with your own SAE. This path works with any framework as long as the functions implement the stated contract. The toy here is a wiring test, not evidence about pretrained language models.

In [ ]:
# EXAMPLE: an SAE with identity decoder and six independent feature coordinates.
# In your real SAE, decoder rows are its learned decoder directions.
decoder = np.eye(6, dtype=float)

def encode_my_sae(hidden_activation):
    return np.asarray(hidden_activation)   # replace with your trained SAE's encoder

my_sae = FunctionalSAE(encode_fn=encode_my_sae, decoder=decoder)

# EXAMPLE: a toy model with a nonlinear downstream interaction between 0 and 2.
# A real model adapter must capture its hidden state at the matching layer/token,
# and score a BATCH of patched hidden states by running the remaining computation.
hidden = np.array([1.5, 1.0, 0.8, 0.6, 0.4, 0.3], dtype=float)

def capture_my_model(prompt):
    return ActivationSite(activation=hidden.copy(), context={"prompt": prompt})

def score_my_model(site, patched_batch):
    assert patched_batch.ndim == 2
    return patched_batch.sum(axis=1) + 1.1*patched_batch[:, 0]*patched_batch[:, 2]

my_model = FunctionalModelAdapter(capture=capture_my_model, score=score_my_model)
custom_result = evaluate_sae(my_model, my_sae, prompt="synthetic",
                             K=6, M=384, n_fit=256, seed=0)
print(f"Custom adapter: SAR1={custom_result.metrics['sar1']:.3f}, "
      f"SAR2={custom_result.metrics['sar2']:.6f}, "
      f"rank={custom_result.metrics['rank1']:.3f} -> {custom_result.metrics['rank2']:.3f}")
assert custom_result.metrics['sar1'] > 0.01
assert custom_result.metrics['sar2'] < 1e-16


## 4 · Integration checklist for your own trained SAE

Before interpreting your scores, verify all of the following:

- **Same model and site:** same base model/checkpoint, layer, pre/post residual hook, token position and SAE input normalization as used to train the SAE. `W_dec` must have shape `(n_features, d_model)`. Passing a tensor with the right dimensions is *not* sufficient to establish a match.
- **SAE reconstruction remainder preserved:** the patched state must be `h - sum(delta_i for removed selected features)`; do not replace it with a reconstruction from only the kept features.
- **Matching downstream function:** return **one scalar per coalition**, from the frozen model with only the specified activation changed. Logit difference is one example; for a different model write your own scoring callable.
- **Holdout and feature cap:** `K` and coalition sample count determine the number of terms. A second-order fit needs at least `1 + K + K*(K-1)/2` fit rows and preferably more independent samples.
- **No implicit universality:** `HookedTransformerAdapter` supports local TransformerLens intervention hooks with vector-shaped activations. For ordinary Hugging Face models and new architectures, implement `FunctionalModelAdapter` with your own capture/replay callbacks, or a verified compatible wrapper.
- **Inspect the evidence:** `result.masks`, `result.responses`, `result.fit_indices`, `result.held_indices`, `result.contributions`, `result.feature_ids` and `result.full_response` are available for auditing.

If the full-coalition replay does not agree with the original model output, stop and fix the site or replay implementation. See [`sar_runner.py`](../sar_runner.py) and [`tests/test_sar_runner.py`](../tests/test_sar_runner.py) for executable reference contracts.

**Interpretation:** SAR is specific to the chosen prompts, behavior, feature selection, intervention and coalition distribution, not a certified global quality score for an SAE.